# 09. LoRA (Low-Rank Adaptation) & QLoRA Deep Dive

**LoRA** (Hu et al., Microsoft 2021) freezes the pre-trained weight matrix $\mathbf{W}_0 \in \mathbb{R}^{d \times k}$ and decomposes weight updates into two low-rank matrices $\mathbf{B} \in \mathbb{R}^{d \times r}$ and $\mathbf{A} \in \mathbb{R}^{r \times k}$ with rank $r \ll \min(d, k)$:
$$\mathbf{W} = \mathbf{W}_0 + \Delta \mathbf{W} = \mathbf{W}_0 + \frac{\alpha}{r} (\mathbf{B} \cdot \mathbf{A})$$

### QLoRA (Dettmers et al., 2023):
Quantizes frozen base model weights into **4-bit NormalFloat (NF4)** and uses Double Quantization + Paged Optimizers, enabling fine-tuning a 70B model on a single 48GB GPU!

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, lora_alpha=16):
        super().__init__()
        # Frozen pre-trained linear layer
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad = False
        
        # Trainable low-rank adapters
        self.rank = rank
        self.scaling = lora_alpha / rank
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

    def forward(self, x):
        # Standard frozen pass + Low-rank delta
        base_out = self.linear(x)
        lora_delta = (x @ self.lora_A.t() @ self.lora_B.t()) * self.scaling
        return base_out + lora_delta

lora_layer = LoRALinear(in_features=4096, out_features=4096, rank=8)
base_params = 4096 * 4096
trainable_lora_params = sum(p.numel() for p in [lora_layer.lora_A, lora_layer.lora_B])
print(f"Frozen Base Weights:     {base_params:,}")
print(f"Trainable LoRA Weights:  {trainable_lora_params:,}")
print(f"Trainable Ratio:         {(trainable_lora_params / base_params) * 100:.2f}% (99.6% parameter reduction!)")

## 🎯 Exercises & Practice
1. Why is matrix $\mathbf{B}$ initialized to all zeros while matrix $\mathbf{A}$ is initialized with Gaussian noise?

In [ ]:
# Exercise Solution
# Answer: Initializing B = 0 ensures Delta W = B * A = 0 at step 0 of fine-tuning.
# The model starts with its exact pre-trained behavior without disruption!